# **Welcome to the Flux tutorial at ISC High Performance 2025!**
<h1><center><img src="https://wihobbs.github.io/flux%20at%20isc.png" style="max-width: 1000px;" alt="Flux at ISC 2025 merged logos" /></center></h1>

Thanks for joining us today! We're stoked to be sharing an overview, resources, tips and tricks about Flux with you.

### **Tutorial overview**

* **Chapter 1**: Getting Started with Flux, Job Submission and Workflow Examples
  * Flux documentation overview and pointers
  * `flux resource` and resource discovery
  * `flux start`, `flux run`, `flux submit`, `flux batch`, `flux alloc`
  * Tuning and optional arguments for submission commands
* **Chapter 2 (You are here!⭐)**:  Flux's Python API
  * `python3 -c "import flux"` (Flux's python bindings)
  * Synchronous job submission and monitoring
  * the FluxExecutor
* **Chapter 3**: Working within a Flux instance, manipulating and monitoring jobs
  * Monitoring job status/state: `flux job info`, `flux job`, `flux job last`, customizable output
  * JournalConsumer interface from python
  * Flux's config file (applicable to batch and alloc instances)

In [2]:
import flux, flux.job
handle = flux.Flux()
consumer = flux.job.JournalConsumer(handle).start()
while True:
    print(consumer.poll(timeout=-1))

PermissionError: [Errno 1] Request requires owner credentials

#### **Stop writing your own scheduler!** It's hard!

Within a batch job allocation, you can unload and reload the `sched-fluxion-resource` module, allowing you to change the scheduling policy on the fly! The scheduler can reload and completely recover state, running jobs are unaffected.

For an idea of different match policies to try, see [flux-config-sched-fluxion-resource(5)](https://flux-framework.readthedocs.io/projects/flux-sched/en/latest/man5/flux-config-sched-fluxion-resource.html#match-policies).

In [2]:
!flux module unload sched-fluxion-resource
!flux module unload sched-fluxion-qmanager

flux-module: remove sched-fluxion-qmanager: No such file or directory


In [3]:
!flux module load sched-fluxion-resource match-policy=low
!flux module load sched-fluxion-qmanager

#### **KVS Python API**

The Flux Key-Value Store is an essential building block for Flux services, but is available to all instance users for job data as well. Like all data storage mechanisms, it has appropriate use cases and antipatterns as well. [See here for documentation on when you might want to use the KVS, and when you might want to look elsewhere.](https://github.com/flux-framework/flux-core/discussions/5784) Someone at LLNL made a promise to bring donuts to work one day, which inspired the example below.

In [2]:
from functools import partial

import flux
import flux.future
import flux.kvs

handle = flux.Flux()

def we_await_donuts(future):
    print("We await donuts")
    with flux.kvs.KVSTxn(flux_handle=handle) as kt:
        kt.mkdir("donuts")
        kt.put("donuts.old_fashioned", "best")
        kt.put("donuts.plain_raised", "good ol' standby")
        kt.put("donuts.apple_fritter", "excellent alternative")

def samir_brought_donuts(hand, watcher, revents, args, future):
    ## Fetch donut ratings from the flux key-value store
    donuts = flux.kvs.KVSDir(hand, ".donuts")
    for donut in donuts.files():
        print(f'{donut}: {flux.kvs.get(hand, donuts.key_at(donut))}')
    future.fulfill()

def donuts_are_here(future):
    print("The donut future has been fulfilled")

## Create a future for the donuts on our flux handle
donut_future = flux.future.FutureExt(we_await_donuts, flux_handle=handle)

## When the donut future (promise) is fulfilled, we tell everyone the donuts are here
donut_future.then(donuts_are_here)

## We schedule the donut future to be fulfilled in 1 minute 
donut_watcher = handle.timer_watcher_create(6, partial(samir_brought_donuts, future=donut_future))

donut_watcher.start()

handle.reactor_run()

We await donuts
apple_fritter: excellent alternative
old_fashioned: best
plain_raised: good ol' standby
The donut future has been fulfilled


0

And now, since we committed the data to the KVS, we can query it on the command line, too.

In [11]:
!flux kvs ls .

admin       donuts      resource


In [18]:
!flux kvs get .donuts.apple_fritter

"excellent alternative"
